In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

df = pd.read_csv(r'C:\Users\colet\Documents\GEOG5890_Capstone\data\Final_Monthly_Crash_Grid_Table\new\Complete_Grid_Monthly_Data.csv')

predictors = ['avg_Temp', 'avg_WindSpeed', 'avg_SolZenAng', 'Road_Density', 'Intersection_Density', 'LU_Residential_pct', 'LU_Industrial_Transport_pct',
              'LU_Mixed_Use_pct', 'LU_Public_Cultural_Rec_pct', 'LU_Parking_pct', 'LU_Vacant_Agriculture_pct', 'LU_Commercial_Office_pct']

target = 'CrashCount'

In [2]:
train = df[df['Year'] <= 2022].copy() # ~80%
test = df[df['Year'] >= 2023].copy() # ~20%

X_train = train[predictors]
y_train = train[target]

X_test = test[predictors]
y_test = test[target]

In [3]:
rf = RandomForestRegressor(n_estimators = 500, random_state = 42, n_jobs = -1)

rf.fit(X_train, y_train)

print('Trained successfully.')

Trained successfully.


In [4]:
y_pred = rf.predict(X_test)
y_pred = np.maximum(y_pred, 0)

In [5]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print('MAE:', mae)
print('RMSE:', rmse)

MAE: 0.24059826434444925
RMSE: 0.5891754833053165


In [6]:
importance = pd.DataFrame({'Variable': predictors, 
                           'Importance': rf.feature_importances_})

importance = importance.sort_values('Importance', ascending = False)

print(importance)

                       Variable  Importance
3                  Road_Density    0.197741
1                 avg_WindSpeed    0.133384
2                 avg_SolZenAng    0.118686
0                      avg_Temp    0.115815
4          Intersection_Density    0.102421
9                LU_Parking_pct    0.080423
5            LU_Residential_pct    0.075659
11     LU_Commercial_Office_pct    0.054420
10    LU_Vacant_Agriculture_pct    0.046928
8    LU_Public_Cultural_Rec_pct    0.032531
6   LU_Industrial_Transport_pct    0.022745
7              LU_Mixed_Use_pct    0.019246


Get Average Monthly Predicted Crashes

In [11]:
test = test.copy()

test['Predicted_Crashes'] = y_pred

In [18]:
grid_predictions = (test.groupby('GRID_ID')['Predicted_Crashes'].mean().reset_index())

print(grid_predictions.head())

   GRID_ID  Predicted_Crashes
0        1           0.001706
1        2           0.692917
2        3           0.198167
3        4           0.091750
4        5           0.001748


In [19]:
grid_predictions.to_csv('Denver_Grid_Predictions.csv', index = False)

Get Predictions by Month

In [7]:
test = df[df['Year'] >= 2023].copy()

X_test = test[predictors]

test['Predicted_Crashes'] = rf.predict(X_test)

In [8]:
output_csv = r'C:\Users\colet\Documents\GEOG5890_Capstone\Model_part_1\Predictions_by_Month.csv'

test[['GRID_ID', 'Year', 'Month', 'CrashCount', 'Predicted_Crashes']].to_csv(output_csv, index = False)